# CardioM3Net — Full Training Pipeline
## Multimodal Meta-Learning Framework for Cardiovascular Disease Diagnosis

**Architecture:** ECG Encoder (ResNet1D + Attention) → Clinical Encoder (MLP) → Cross-Attention Fusion → Multi-Task Heads

**Training Pipeline:**
1. Self-Supervised Pretraining (SimCLR on ECG)
2. Multi-Task Supervised Training (Binary + Disease + Severity) with Domain Adaptation
3. MAML Meta-Learning (fast cross-dataset adaptation)
4. Federated Learning (privacy-preserving multi-hospital simulation)
5. Explainability (SHAP + Grad-CAM + Modality Attention)

**Datasets:** PTB-XL (21,837 ECGs) + UCI Heart Disease (clinical features)

---
### Setup: Add Datasets and Code
1. Right sidebar → **Add Data** → search `m0hamedyousry/ptb-xl-a-large-publicly-available-ecg-dataset`
2. Right sidebar → **Add Data** → search `redwankarimsony/heart-disease-data`
3. Right sidebar → **Add Data** → **Your Work** → select your uploaded dataset that contains:
   - `train_cardiom3net.py`
   - `cardiom3net/` (entire folder with ALL subfolders including `__init__.py` files)
4. Right sidebar → **Settings** → **Accelerator** → **GPU T4 x2**

> **Tip:** When uploading `cardiom3net/` to a Kaggle dataset, use **zip upload** or the Kaggle API
> (`kaggle datasets init -p .` in the project root) to ensure hidden files like `__init__.py` are included.


In [ ]:
# STEP 1: Install dependencies
!pip install -q wfdb shap

In [ ]:
# STEP 2: Copy cardiom3net package from Kaggle input dataset → /kaggle/working/
# This is REQUIRED because Python imports need the package to be in a writable,
# properly-resolved path. The /kaggle/input/ directory can cause import issues.

import os
import shutil
import sys

WORKING = '/kaggle/working'

def _find_and_copy_package(pkg_name, src_root='/kaggle/input', dst_root=WORKING):
    """Walk src_root to find pkg_name/ and copy it to dst_root."""
    dst = os.path.join(dst_root, pkg_name)
    if os.path.isdir(dst):
        print(f"[OK] {pkg_name}/ already at {dst}")
        return dst
    for root, dirs, _ in os.walk(src_root):
        if pkg_name in dirs:
            src = os.path.join(root, pkg_name)
            shutil.copytree(src, dst)
            print(f"[COPIED] {src}  →  {dst}")
            return dst
    print(f"[ERROR] '{pkg_name}/' not found under {src_root}")
    return None

def _find_and_copy_file(filename, src_root='/kaggle/input', dst_root=WORKING):
    """Find filename under src_root and copy to dst_root."""
    dst = os.path.join(dst_root, filename)
    if os.path.isfile(dst):
        print(f"[OK] {filename} already at {dst}")
        return dst
    for root, _, files in os.walk(src_root):
        if filename in files:
            src = os.path.join(root, filename)
            shutil.copy2(src, dst)
            print(f"[COPIED] {src}  →  {dst}")
            return dst
    print(f"[ERROR] '{filename}' not found under {src_root}")
    return None

# Copy package and script
pkg_dst  = _find_and_copy_package('cardiom3net')
script   = _find_and_copy_file('train_cardiom3net.py')

# Add working dir to Python path
if WORKING not in sys.path:
    sys.path.insert(0, WORKING)

# ── Verify all __init__.py files are present ────────────────────────────────
print("\n── Package structure verification ──")
required = [
    'cardiom3net/__init__.py',
    'cardiom3net/config.py',
    'cardiom3net/utils.py',
    'cardiom3net/data/__init__.py',
    'cardiom3net/data/ecg_loader.py',
    'cardiom3net/data/clinical_loader.py',
    'cardiom3net/data/augmentation.py',
    'cardiom3net/data/multimodal_dataset.py',
    'cardiom3net/models/__init__.py',
    'cardiom3net/models/ecg_encoder.py',
    'cardiom3net/models/clinical_encoder.py',
    'cardiom3net/models/fusion.py',
    'cardiom3net/models/multitask_head.py',
    'cardiom3net/models/domain_adaptation.py',
    'cardiom3net/models/cardiom3net.py',
    'cardiom3net/training/__init__.py',
    'cardiom3net/training/self_supervised.py',
    'cardiom3net/training/supervised.py',
    'cardiom3net/training/maml_trainer.py',
    'cardiom3net/training/federated.py',
    'cardiom3net/explainability/__init__.py',
    'cardiom3net/explainability/gradcam_1d.py',
    'cardiom3net/explainability/shap_analysis.py',
]
all_ok = True
for rel in required:
    full = os.path.join(WORKING, rel)
    status = '[OK]' if os.path.isfile(full) else '[MISSING]'
    if status == '[MISSING]':
        all_ok = False
    print(f"  {status}  {rel}")

if all_ok:
    print("\n✓ All package files verified.")
    # Quick import test
    from cardiom3net.models.cardiom3net import CardioM3Net
    print("✓ Import test passed: CardioM3Net importable.")
else:
    print("\n✗ Some files are missing — re-upload the cardiom3net/ folder to your Kaggle dataset.")


In [ ]:
# STEP 3: Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# STEP 4: Run FULL CardioM3Net training pipeline
# Runs ALL phases: SSL → Supervised → MAML → Federated → Explainability
# Script runs from /kaggle/working/ where cardiom3net/ is already copied.

!python /kaggle/working/train_cardiom3net.py \
    --epochs 30 \
    --ssl_epochs 20 \
    --batch_size 32 \
    --lr 0.001 \
    --maml_episodes 200 \
    --fed_rounds 5 \
    --fed_clients 3 \
    --output_dir /kaggle/working/cardiom3net_results


In [ ]:
# STEP 5: Quick run (skip SSL and MAML for faster testing)
# Uncomment and run this instead of STEP 4 for a fast smoke-test:

# !python /kaggle/working/train_cardiom3net.py \
#     --epochs 10 \
#     --batch_size 64 \
#     --skip_ssl \
#     --skip_maml \
#     --fed_rounds 3 \
#     --output_dir /kaggle/working/cardiom3net_results_quick


In [ ]:
# STEP 6: View results
import os
from IPython.display import Image, display

results_dir = '/kaggle/working/cardiom3net_results'

print('Generated files:')
for f in sorted(os.listdir(results_dir)):
    size = os.path.getsize(os.path.join(results_dir, f))
    print(f'  {f} ({size/1024:.0f} KB)')

# Display plots
for img in ['training_curves.png', 'confusion_matrices.png', 'roc_curves.png',
            'ecg_saliency.png', 'shap_clinical.png', 'modality_weights.png']:
    path = os.path.join(results_dir, img)
    if os.path.exists(path):
        print(f'\n--- {img} ---')
        display(Image(filename=path, width=800))

In [ ]:
# STEP 7: Load and inspect saved model
import torch

checkpoint = torch.load('/kaggle/working/cardiom3net_results/cardiom3net_central.pth',
                        map_location='cpu')

print('Model config:')
for k, v in checkpoint['config'].items():
    print(f'  {k}: {v}')

print('\nResults:')
for task, metrics in checkpoint['results'].items():
    print(f'  {task}:')
    for m, v in metrics.items():
        print(f'    {m}: {v:.4f}')